In [1]:
import pandas as pd
import numpy as np

# Load the dataset we saved yesterday
df = pd.read_csv('../data/inventory_sales_data.csv', parse_dates=['Date'])

print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
df.head()

Rows: 36550, Columns: 15


,Product,Category,Warehouse,Supplier,Date,Season,Promotion,Price,Lead_Time,Raw_Demand,Historical_Sales,Returns,Current_Inventory,Lost_Sales,Stockout_Flag
0,P001,Electronics,WH_North,Supplier_C,2023-01-01,Winter,0,12714.12,5,11,11,0,143,0,0
1,P001,Electronics,WH_North,Supplier_C,2023-01-02,Winter,0,12714.12,5,11,11,0,132,0,0
2,P001,Electronics,WH_North,Supplier_C,2023-01-03,Winter,0,12714.12,5,11,11,0,121,0,0
3,P001,Electronics,WH_North,Supplier_C,2023-01-04,Winter,0,12714.12,5,13,13,0,108,0,0
4,P001,Electronics,WH_North,Supplier_C,2023-01-05,Winter,0,12714.12,5,16,16,0,92,0,0


In [2]:
# ---- VALIDATION CHECKS ----

print("=== Duplicate rows ===")
print(f"Exact duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate Product+Date combos: {df.duplicated(subset=['Product','Date']).sum()}")

print("\n=== Missing values ===")
print(df.isnull().sum().sum(), "total missing values")

print("\n=== Negative value checks (should all be 0) ===")
print(f"Negative Historical_Sales: {(df['Historical_Sales'] < 0).sum()}")
print(f"Negative Current_Inventory: {(df['Current_Inventory'] < 0).sum()}")
print(f"Negative Price: {(df['Price'] < 0).sum()}")

print("\n=== Logical consistency ===")
print(f"Rows where Historical_Sales > Raw_Demand: {(df['Historical_Sales'] > df['Raw_Demand']).sum()}")
print(f"Rows where Lost_Sales != Raw_Demand - Historical_Sales: {(df['Lost_Sales'] != (df['Raw_Demand'] - df['Historical_Sales'])).sum()}")

print("\n=== Date coverage per product ===")
date_counts = df.groupby('Product')['Date'].count()
print(f"Expected days per product: 731")
print(f"Products with wrong day count: {(date_counts != 731).sum()}")

=== Duplicate rows ===
Exact duplicate rows: 0
Duplicate Product+Date combos: 0

=== Missing values ===
0 total missing values

=== Negative value checks (should all be 0) ===
Negative Historical_Sales: 0
Negative Current_Inventory: 0
Negative Price: 0

=== Logical consistency ===
Rows where Historical_Sales > Raw_Demand: 0
Rows where Lost_Sales != Raw_Demand - Historical_Sales: 0

=== Date coverage per product ===
Expected days per product: 731
Products with wrong day count: 0


In [3]:
# ---- FEATURE 1: ROLLING AVERAGE DEMAND (7-day and 30-day) ----

# Sort first - rolling calculations MUST go in date order per product
df = df.sort_values(['Product', 'Date']).reset_index(drop=True)

# groupby('Product') ensures the rolling window never "leaks" across different products
df['Rolling_7d_Demand'] = df.groupby('Product')['Raw_Demand'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

df['Rolling_30d_Demand'] = df.groupby('Product')['Raw_Demand'].transform(
    lambda x: x.rolling(window=30, min_periods=1).mean()
)

df[df['Product']=='P001'][['Date','Raw_Demand','Rolling_7d_Demand','Rolling_30d_Demand']].head(15)

,Date,Raw_Demand,Rolling_7d_Demand,Rolling_30d_Demand
0,2023-01-01,11,11.000000,11.000000
1,2023-01-02,11,11.000000,11.000000
2,2023-01-03,11,11.000000,11.000000
3,2023-01-04,13,11.500000,11.500000
4,2023-01-05,16,12.400000,12.400000
5,2023-01-06,16,13.000000,13.000000
6,2023-01-07,10,12.571429,12.571429
7,2023-01-08,13,12.857143,12.625000
8,2023-01-09,17,13.714286,13.111111
9,2023-01-10,11,13.714286,12.900000


In [4]:
# ---- FEATURE 2: DAYS OF STOCK REMAINING ----

# Avoid division by zero: if rolling demand is 0, treat "days remaining" as a large number (effectively "safe")
df['Days_Stock_Remaining'] = np.where(
    df['Rolling_7d_Demand'] > 0,
    df['Current_Inventory'] / df['Rolling_7d_Demand'],
    999  # placeholder for "no recent demand, not at risk"
)

df[df['Product']=='P001'][['Date','Current_Inventory','Rolling_7d_Demand','Days_Stock_Remaining']].head(15)

,Date,Current_Inventory,Rolling_7d_Demand,Days_Stock_Remaining
0,2023-01-01,143,11.000000,13.000000
1,2023-01-02,132,11.000000,12.000000
2,2023-01-03,121,11.000000,11.000000
3,2023-01-04,108,11.500000,9.391304
4,2023-01-05,92,12.400000,7.419355
5,2023-01-06,76,13.000000,5.846154
6,2023-01-07,66,12.571429,5.250000
7,2023-01-08,53,12.857143,4.122222
8,2023-01-09,36,13.714286,2.625000
9,2023-01-10,25,13.714286,1.822917


In [5]:
# ---- FEATURE 3: EXPECTED DEMAND DURING LEAD TIME + STOCKOUT RISK ----

# Expected demand while waiting for a reorder to arrive
df['Expected_Demand_During_LeadTime'] = df['Rolling_7d_Demand'] * df['Lead_Time']

# Risk flag: is current inventory enough to cover demand during the lead time?
df['At_Risk_Flag'] = (df['Current_Inventory'] < df['Expected_Demand_During_LeadTime']).astype(int)

# How much shortfall (or surplus) we have, in units
df['Inventory_Gap'] = df['Current_Inventory'] - df['Expected_Demand_During_LeadTime']

df[df['Product']=='P001'][['Date','Current_Inventory','Lead_Time','Expected_Demand_During_LeadTime','At_Risk_Flag','Inventory_Gap']].head(15)

,Date,Current_Inventory,Lead_Time,Expected_Demand_During_LeadTime,At_Risk_Flag,Inventory_Gap
0,2023-01-01,143,5,55.000000,0,88.000000
1,2023-01-02,132,5,55.000000,0,77.000000
2,2023-01-03,121,5,55.000000,0,66.000000
3,2023-01-04,108,5,57.500000,0,50.500000
4,2023-01-05,92,5,62.000000,0,30.000000
5,2023-01-06,76,5,65.000000,0,11.000000
6,2023-01-07,66,5,62.857143,0,3.142857
7,2023-01-08,53,5,64.285714,1,-11.285714
8,2023-01-09,36,5,68.571429,1,-32.571429
9,2023-01-10,25,5,68.571429,1,-43.571429


In [6]:
# ---- SAVE CLEANED + ENRICHED DATASET ----

output_path = '../data/inventory_data_cleaned.csv'
df.to_csv(output_path, index=False)

print(f"Saved {len(df)} rows, {len(df.columns)} columns to {output_path}")
print(f"\nNew columns added today: Rolling_7d_Demand, Rolling_30d_Demand, Days_Stock_Remaining,")
print(f"Expected_Demand_During_LeadTime, At_Risk_Flag, Inventory_Gap")

print(f"\nOverall At_Risk_Flag rate: {df['At_Risk_Flag'].mean()*100:.1f}% of product-days currently flagged at-risk")

Saved 36550 rows, 21 columns to ../data/inventory_data_cleaned.csv

New columns added today: Rolling_7d_Demand, Rolling_30d_Demand, Days_Stock_Remaining,
Expected_Demand_During_LeadTime, At_Risk_Flag, Inventory_Gap

Overall At_Risk_Flag rate: 79.3% of product-days currently flagged at-risk
